# Ledger — Colab Runner (fixed)

This notebook runs your **Ledger** FastAPI backend (`main.py`, `document_processors.py`, `utils.py`) inside Colab, tunnels it publicly with ngrok, and (optionally) pushes the live URL to your GitHub repo so your static frontend can find it.

**What was broken in the old notebook:** every cell was running `exec(open('RANDOM_TOKEN','r').read())` where `RANDOM_TOKEN` was a placeholder that never got replaced with real file content or a real path — so Colab was trying to open a file that didn't exist. This notebook writes your **actual** source files to disk and imports them normally instead.

**Two bugs fixed from the original `colab_ledger_runner.py`:**
1. `uvicorn` was never installed (only `requirements.txt` + `pyngrok` + `requests` were) even though Cell 5 does `import uvicorn` — added it to the install step below.
2. A hardcoded string that looked like a leaked API key (`API_KEY = 'SA40022791rah$_$'`) was sitting unused in the original middleware cell — it's dropped here. **If that was a real key, rotate it**, since your own README notes this repo is public.

**Before running:** open the key icon (🔑) in the left sidebar → add these Colab secrets → toggle "Notebook access" on for each:
- `GROQ_API_KEY`
- `WEAVIATE_URL`
- `WEAVIATE_API_KEY`
- `LEDGER_SHARED_SECRET` (any random string you make up — this is the header your frontend must send)
- `NGROK_AUTH_TOKEN` (free, from https://dashboard.ngrok.com/get-started/your-authtoken)
- `GITHUB_PAT` (only needed if you want Cell 8 to auto-push `api-config.json` to GitHub)

If a secret isn't set, Cell 3 below will prompt you for it instead of crashing, so you can still run everything and test it end to end.

In [ ]:
# --- Cell 1: install dependencies -----------------------------------------
# Fixed: uvicorn was missing from the original install command.
!pip install -q fastapi==0.115.0 python-multipart==0.0.9 pydantic==2.9.2 \
    weaviate-client==4.9.6 groq==0.11.0 pymupdf==1.24.10 python-pptx==1.0.2 \
    Pillow==10.4.0 tabulate==0.9.0 pandas==2.2.3 uvicorn pyngrok requests
print("Dependencies installed.")

In [ ]:
# --- Cell 2: write the real source files to disk ---------------------------
# These are your actual utils.py / document_processors.py / main.py contents,
# not placeholder tokens. Each is a JSON-escaped string literal so escaping
# round-trips byte-for-byte (verified against your originals before shipping
# this notebook).
import os

APP_DIR = "/content/ledger_api"
os.makedirs(APP_DIR, exist_ok=True)

files = {
    "utils.py": "# utils.py\n#\n# Lightweight, serverless-friendly helpers for the Multimodal Financial RAG API.\n#\n# Design notes (important for Vercel deployment):\n# - No torch / transformers / local model weights. Vision, text, and speech\n#   all go through the Groq Cloud API, which returns in a fraction of a\n#   second and keeps the deployed function small enough to fit Vercel's\n#   serverless size limits.\n# - Every function here is pure / stateless: given bytes in, text out. No\n#   files are written to disk anywhere in this module.\n# - Model names are read from environment variables with sensible defaults\n#   so you can swap models the moment Groq ships something newer, without a\n#   redeploy of code (only an env var change).\n\nimport os\nimport base64\nfrom functools import lru_cache\nfrom typing import List, Tuple\n\nimport fitz  # PyMuPDF\nfrom groq import Groq\n\n# ---------------------------------------------------------------------------\n# Groq client + model configuration\n# ---------------------------------------------------------------------------\n\nVISION_MODEL = os.environ.get(\"GROQ_VISION_MODEL\", \"llama-3.2-11b-vision-preview\")\nTEXT_MODEL = os.environ.get(\"GROQ_TEXT_MODEL\", \"llama-3.3-70b-versatile\")\nWHISPER_MODEL = os.environ.get(\"GROQ_WHISPER_MODEL\", \"whisper-large-v3\")\n\n\n@lru_cache(maxsize=1)\ndef get_groq_client() -> Groq:\n    \"\"\"Lazily create a single Groq client per warm serverless instance.\"\"\"\n    api_key = os.environ.get(\"GROQ_API_KEY\")\n    if not api_key:\n        raise RuntimeError(\n            \"GROQ_API_KEY is not set. Add it in your Vercel project's \"\n            \"Environment Variables settings.\"\n        )\n    return Groq(api_key=api_key)\n\n\n# ---------------------------------------------------------------------------\n# Vision (image / chart / table description)\n# ---------------------------------------------------------------------------\n\ndef _image_bytes_to_data_url(image_bytes: bytes, mime: str = \"image/png\") -> str:\n    b64 = base64.b64encode(image_bytes).decode(\"utf-8\")\n    return f\"data:{mime};base64,{b64}\"\n\n\ndef describe_image(\n    image_bytes: bytes,\n    prompt: str = (\n        \"Describe what you see in this image in detail, including any \"\n        \"visible text, numbers, labels, or data.\"\n    ),\n) -> str:\n    \"\"\"Send an image straight from memory to Groq's vision model.\"\"\"\n    client = get_groq_client()\n    data_url = _image_bytes_to_data_url(image_bytes)\n    response = client.chat.completions.create(\n        model=VISION_MODEL,\n        messages=[\n            {\n                \"role\": \"user\",\n                \"content\": [\n                    {\"type\": \"text\", \"text\": prompt},\n                    {\"type\": \"image_url\", \"image_url\": {\"url\": data_url}},\n                ],\n            }\n        ],\n        temperature=0.2,\n        max_tokens=1024,\n    )\n    return response.choices[0].message.content or \"\"\n\n\ndef is_graph(description: str) -> bool:\n    \"\"\"Heuristic: does this image description look like a chart/graph/table?\"\"\"\n    keywords = (\"graph\", \"chart\", \"plot\", \"table\", \"diagram\", \"axis\", \"trend\")\n    lowered = description.lower()\n    return any(k in lowered for k in keywords)\n\n\ndef describe_chart_or_table(image_bytes: bytes) -> str:\n    \"\"\"Ask specifically for the underlying data of a financial chart/table.\"\"\"\n    prompt = (\n        \"This image is a financial chart, graph, or table. Transcribe the \"\n        \"underlying data as precisely as possible: axis labels, series \"\n        \"names, approximate values, trends, and any notable figures. Write \"\n        \"it as plain text suitable for a search index, not prose.\"\n    )\n    return describe_image(image_bytes, prompt=prompt)\n\n\n# ---------------------------------------------------------------------------\n# Speech-to-text\n# ---------------------------------------------------------------------------\n\ndef transcribe_audio(audio_bytes: bytes, filename: str) -> str:\n    \"\"\"Transcribe an earnings-call recording or voice memo via Groq Whisper.\"\"\"\n    client = get_groq_client()\n    transcription = client.audio.transcriptions.create(\n        model=WHISPER_MODEL,\n        file=(filename, audio_bytes),\n        response_format=\"text\",\n    )\n    if isinstance(transcription, str):\n        return transcription\n    return getattr(transcription, \"text\", str(transcription))\n\n\n# ---------------------------------------------------------------------------\n# LLM chat completion (RAG answer generation)\n# ---------------------------------------------------------------------------\n\ndef chat_completion(\n    system_prompt: str,\n    user_prompt: str,\n    temperature: float = 0.2,\n    max_tokens: int = 1024,\n) -> str:\n    client = get_groq_client()\n    response = client.chat.completions.create(\n        model=TEXT_MODEL,\n        messages=[\n            {\"role\": \"system\", \"content\": system_prompt},\n            {\"role\": \"user\", \"content\": user_prompt},\n        ],\n        temperature=temperature,\n        max_tokens=max_tokens,\n    )\n    return response.choices[0].message.content or \"\"\n\n\n# ---------------------------------------------------------------------------\n# Text chunking (replaces LangChain's RecursiveCharacterTextSplitter so we\n# don't have to pull in the full LangChain dependency tree)\n# ---------------------------------------------------------------------------\n\ndef chunk_text(text: str, chunk_size: int = 600, overlap: int = 50) -> List[str]:\n    text = (text or \"\").strip()\n    if not text:\n        return []\n    if len(text) <= chunk_size:\n        return [text]\n\n    chunks = []\n    start = 0\n    step = max(1, chunk_size - overlap)\n    while start < len(text):\n        end = start + chunk_size\n        chunk = text[start:end].strip()\n        if chunk:\n            chunks.append(chunk)\n        start += step\n    return chunks\n\n\n# ---------------------------------------------------------------------------\n# PDF layout helpers (pure functions, no external services)\n# ---------------------------------------------------------------------------\n\ndef extract_text_around_item(\n    text_blocks, bbox, page_height, threshold_percentage: float = 0.1\n) -> Tuple[str, str]:\n    \"\"\"Find the text block immediately above/below a bounding box (e.g. a\n    table or image) so we can use it as a caption hint.\"\"\"\n    before_text, after_text = \"\", \"\"\n    vertical_threshold_distance = page_height * threshold_percentage\n    horizontal_threshold_distance = bbox.width * threshold_percentage\n\n    for block in text_blocks:\n        block_bbox = fitz.Rect(block[:4])\n        vertical_distance = min(\n            abs(block_bbox.y1 - bbox.y0), abs(block_bbox.y0 - bbox.y1)\n        )\n        horizontal_overlap = max(\n            0, min(block_bbox.x1, bbox.x1) - max(block_bbox.x0, bbox.x0)\n        )\n\n        if (\n            vertical_distance <= vertical_threshold_distance\n            and horizontal_overlap >= -horizontal_threshold_distance\n        ):\n            if block_bbox.y1 < bbox.y0 and not before_text:\n                before_text = block[4]\n            elif block_bbox.y0 > bbox.y1 and not after_text:\n                after_text = block[4]\n                break\n\n    return before_text, after_text\n\n\ndef process_text_blocks(text_blocks, char_count_threshold: int = 500):\n    \"\"\"Group adjacent text blocks into passages of roughly char_count_threshold\n    characters, so short headings/paragraphs aren't indexed as isolated\n    fragments.\"\"\"\n    current_group = []\n    grouped_blocks = []\n    current_char_count = 0\n\n    for block in text_blocks:\n        if block[-1] == 0:  # text-type block\n            block_text = block[4]\n            block_char_count = len(block_text)\n\n            if current_char_count + block_char_count <= char_count_threshold:\n                current_group.append(block)\n                current_char_count += block_char_count\n            else:\n                if current_group:\n                    grouped_content = \"\\n\".join(b[4] for b in current_group)\n                    grouped_blocks.append((current_group[0], grouped_content))\n                current_group = [block]\n                current_char_count = block_char_count\n\n    if current_group:\n        grouped_content = \"\\n\".join(b[4] for b in current_group)\n        grouped_blocks.append((current_group[0], grouped_content))\n\n    return grouped_blocks\n",
    "document_processors.py": "# document_processors.py\n#\n# Extracts text/table/image/audio content from uploaded files and returns\n# plain Python dicts ready for chunking + indexing. Everything happens\n# in-memory (BytesIO / bytes) \u2014 nothing is ever written to the serverless\n# function's read-only filesystem.\n\nimport io\nimport os\nfrom typing import List, Tuple, Dict\n\nimport fitz  # PyMuPDF\nfrom pptx import Presentation\n\nfrom utils import (\n    describe_image,\n    is_graph,\n    describe_chart_or_table,\n    transcribe_audio,\n    extract_text_around_item,\n    process_text_blocks,\n)\n\nIMAGE_EXTENSIONS = {\".png\", \".jpg\", \".jpeg\", \".webp\", \".gif\"}\nAUDIO_EXTENSIONS = {\".mp3\", \".wav\", \".m4a\", \".flac\", \".ogg\", \".webm\"}\n\n\ndef process_pdf(file_bytes: bytes, filename: str) -> List[Dict]:\n    \"\"\"Extract text passages, tables, and figures from a PDF, entirely\n    in-memory.\"\"\"\n    docs: List[Dict] = []\n    base_name = os.path.splitext(filename)[0]\n\n    try:\n        pdf = fitz.open(stream=file_bytes, filetype=\"pdf\")\n    except Exception as e:\n        return [{\n            \"text\": f\"[Could not open PDF '{filename}': {e}]\",\n            \"metadata\": {\"source\": filename, \"type\": \"error\", \"page_num\": 0},\n        }]\n\n    for page_num in range(len(pdf)):\n        page = pdf[page_num]\n\n        text_blocks = [\n            b for b in page.get_text(\"blocks\", sort=True)\n            if b[-1] == 0\n            and not (b[1] < page.rect.height * 0.1 or b[3] > page.rect.height * 0.9)\n        ]\n        grouped_text_blocks = process_text_blocks(text_blocks)\n\n        # --- Tables -----------------------------------------------------\n        table_bboxes = []\n        try:\n            tables = page.find_tables(\n                horizontal_strategy=\"lines_strict\", vertical_strategy=\"lines_strict\"\n            )\n            for idx, tab in enumerate(tables, start=1):\n                try:\n                    table_df = tab.to_pandas()\n                    bbox = fitz.Rect(tab.bbox)\n                    table_bboxes.append(bbox)\n\n                    before_text, after_text = extract_text_around_item(\n                        text_blocks, bbox, page.rect.height\n                    )\n                    columns = \", \".join(str(c) for c in table_df.columns)\n                    caption = (before_text + \" \" + after_text).strip()\n                    if not caption:\n                        caption = columns\n\n                    try:\n                        table_repr = table_df.to_markdown(index=False)\n                    except Exception:\n                        table_repr = table_df.to_csv(index=False)\n\n                    docs.append({\n                        \"text\": (\n                            f\"This is a table with caption: {caption}\\n\"\n                            f\"Columns: {columns}\\n{table_repr}\"\n                        ),\n                        \"metadata\": {\n                            \"source\": f\"{base_name}-page{page_num}-table{idx}\",\n                            \"type\": \"table\",\n                            \"page_num\": page_num,\n                        },\n                    })\n                except Exception:\n                    continue\n        except Exception:\n            pass\n\n        # --- Images / charts ---------------------------------------------\n        try:\n            for image_info in page.get_image_info(xrefs=True):\n                xref = image_info.get(\"xref\", 0)\n                if xref == 0:\n                    continue\n\n                img_bbox = fitz.Rect(image_info[\"bbox\"])\n                if (\n                    img_bbox.width < page.rect.width / 20\n                    or img_bbox.height < page.rect.height / 20\n                ):\n                    continue\n\n                extracted = pdf.extract_image(xref)\n                image_bytes = extracted[\"image\"]\n\n                before_text, after_text = extract_text_around_item(\n                    text_blocks, img_bbox, page.rect.height\n                )\n                caption_hint = (before_text + \" \" + after_text).strip()\n\n                try:\n                    description = describe_image(image_bytes)\n                    if is_graph(description):\n                        description = describe_chart_or_table(image_bytes)\n                except Exception as e:\n                    description = f\"[Image description unavailable: {e}]\"\n\n                docs.append({\n                    \"text\": (\n                        f\"This is an image on page {page_num}\"\n                        + (f\" (context: {caption_hint})\" if caption_hint else \"\")\n                        + f\". Description: {description}\"\n                    ),\n                    \"metadata\": {\n                        \"source\": f\"{base_name}-page{page_num}-image{xref}\",\n                        \"type\": \"image\",\n                        \"page_num\": page_num,\n                    },\n                })\n        except Exception:\n            pass\n\n        # --- Body text ------------------------------------------------------\n        for block_ctr, (heading_block, content) in enumerate(grouped_text_blocks, start=1):\n            heading_bbox = fitz.Rect(heading_block[:4])\n            if any(heading_bbox.intersects(tb) for tb in table_bboxes):\n                continue\n            docs.append({\n                \"text\": f\"{heading_block[4]}\\n{content}\",\n                \"metadata\": {\n                    \"source\": f\"{base_name}-page{page_num}-block{block_ctr}\",\n                    \"type\": \"text\",\n                    \"page_num\": page_num,\n                },\n            })\n\n    pdf.close()\n    return docs\n\n\ndef process_image_file(file_bytes: bytes, filename: str) -> List[Dict]:\n    try:\n        description = describe_image(file_bytes)\n    except Exception as e:\n        description = f\"[Image description unavailable: {e}]\"\n    return [{\n        \"text\": description,\n        \"metadata\": {\"source\": filename, \"type\": \"image\", \"page_num\": 0},\n    }]\n\n\ndef process_pptx_file(file_bytes: bytes, filename: str) -> List[Dict]:\n    \"\"\"Extract slide text + speaker notes from a .pptx file.\n\n    Note: legacy binary .ppt and slide-to-image rendering (which the\n    original app did via a local LibreOffice install) are not supported\n    here, since Vercel's serverless functions have no LibreOffice binary\n    available. Ask users to export legacy .ppt files to .pptx first.\n    \"\"\"\n    docs: List[Dict] = []\n    try:\n        prs = Presentation(io.BytesIO(file_bytes))\n    except Exception as e:\n        return [{\n            \"text\": f\"[Could not open presentation '{filename}': {e}]\",\n            \"metadata\": {\"source\": filename, \"type\": \"error\", \"page_num\": 0},\n        }]\n\n    for slide_num, slide in enumerate(prs.slides):\n        texts = [\n            shape.text for shape in slide.shapes\n            if hasattr(shape, \"text\") and shape.text\n        ]\n        slide_text = \"\\n\".join(texts)\n        try:\n            notes = (\n                slide.notes_slide.notes_text_frame.text\n                if slide.has_notes_slide else \"\"\n            )\n        except Exception:\n            notes = \"\"\n\n        combined = slide_text + (f\"\\n\\nSpeaker notes: {notes}\" if notes else \"\")\n        if combined.strip():\n            docs.append({\n                \"text\": combined,\n                \"metadata\": {\n                    \"source\": f\"{filename}-slide{slide_num + 1}\",\n                    \"type\": \"slide\",\n                    \"page_num\": slide_num,\n                },\n            })\n    return docs\n\n\ndef process_text_file(file_bytes: bytes, filename: str) -> List[Dict]:\n    try:\n        text = file_bytes.decode(\"utf-8\")\n    except UnicodeDecodeError:\n        text = file_bytes.decode(\"latin-1\", errors=\"ignore\")\n    return [{\n        \"text\": text,\n        \"metadata\": {\"source\": filename, \"type\": \"text\", \"page_num\": 0},\n    }]\n\n\ndef process_audio_file(file_bytes: bytes, filename: str) -> List[Dict]:\n    try:\n        transcript = transcribe_audio(file_bytes, filename)\n    except Exception as e:\n        transcript = f\"[Audio transcription unavailable: {e}]\"\n    return [{\n        \"text\": transcript,\n        \"metadata\": {\"source\": filename, \"type\": \"audio\", \"page_num\": 0},\n    }]\n\n\ndef load_multimodal_data(files: List[Tuple[str, bytes]]) -> List[Dict]:\n    \"\"\"Route each uploaded (filename, bytes) pair to the right processor.\n\n    Returns a flat list of {\"text\": ..., \"metadata\": {...}} documents, ready\n    to be chunked and indexed.\n    \"\"\"\n    documents: List[Dict] = []\n    for filename, file_bytes in files:\n        ext = os.path.splitext(filename.lower())[1]\n        try:\n            if ext in IMAGE_EXTENSIONS:\n                documents.extend(process_image_file(file_bytes, filename))\n            elif ext == \".pdf\":\n                documents.extend(process_pdf(file_bytes, filename))\n            elif ext == \".pptx\":\n                documents.extend(process_pptx_file(file_bytes, filename))\n            elif ext in AUDIO_EXTENSIONS:\n                documents.extend(process_audio_file(file_bytes, filename))\n            else:\n                documents.extend(process_text_file(file_bytes, filename))\n        except Exception as e:\n            documents.append({\n                \"text\": f\"[Error processing {filename}: {e}]\",\n                \"metadata\": {\"source\": filename, \"type\": \"error\", \"page_num\": 0},\n            })\n    return documents\n",
    "main.py": "# main.py\n#\n# FastAPI backend for the Multimodal Financial RAG app, designed to run as a\n# single Vercel Serverless Function (see /vercel.json). It is fully\n# stateless: every request opens a short-lived connection to Weaviate Cloud,\n# does its work, and closes it again. No state is kept in memory or on disk\n# between invocations, and no local ML models are loaded \u2014 all embedding,\n# vision, and generation calls go to Weaviate Cloud / Groq Cloud.\n#\n# Required environment variables (set these in the Vercel project settings):\n#   GROQ_API_KEY        - https://console.groq.com\n#   WEAVIATE_URL        - your Weaviate Cloud (WCS) cluster REST endpoint\n#   WEAVIATE_API_KEY    - your Weaviate Cloud API key\n#\n# The Weaviate collection is configured to use Weaviate's own hosted\n# vectorizer (text2vec-weaviate), so no separate embeddings API key is\n# needed. If your cluster doesn't have that module enabled, swap the\n# vectorizer_config in ensure_collection() for text2vec-cohere / -openai\n# and add the matching header/API key.\n\nimport os\nimport uuid\nfrom typing import List, Optional, Dict, Any\n\nfrom fastapi import FastAPI, UploadFile, File, Form, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel\n\nimport weaviate\nfrom weaviate.classes.init import Auth\nfrom weaviate.classes.config import Configure, Property, DataType\nfrom weaviate.classes.query import Filter\n\nfrom document_processors import load_multimodal_data\nfrom utils import chunk_text, chat_completion\n\nCOLLECTION_NAME = \"FinancialRagChunk\"\n\napp = FastAPI(title=\"Multimodal Financial RAG API\")\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=[\"*\"],\n    allow_methods=[\"*\"],\n    allow_headers=[\"*\"],\n)\n\n\n# ---------------------------------------------------------------------------\n# Weaviate connection helpers\n# ---------------------------------------------------------------------------\n\ndef get_weaviate_client():\n    url = os.environ.get(\"WEAVIATE_URL\")\n    api_key = os.environ.get(\"WEAVIATE_API_KEY\")\n    if not url or not api_key:\n        raise HTTPException(\n            status_code=500,\n            detail=\"WEAVIATE_URL / WEAVIATE_API_KEY are not configured on the server.\",\n        )\n    return weaviate.connect_to_weaviate_cloud(\n        cluster_url=url,\n        auth_credentials=Auth.api_key(api_key),\n    )\n\n\ndef ensure_collection(client):\n    if not client.collections.exists(COLLECTION_NAME):\n        client.collections.create(\n            name=COLLECTION_NAME,\n            vectorizer_config=Configure.Vectorizer.text2vec_weaviate(),\n            properties=[\n                Property(name=\"text\", data_type=DataType.TEXT),\n                Property(name=\"source\", data_type=DataType.TEXT),\n                Property(name=\"doc_type\", data_type=DataType.TEXT),\n                Property(name=\"page_num\", data_type=DataType.INT),\n                Property(name=\"session_id\", data_type=DataType.TEXT),\n            ],\n        )\n    return client.collections.get(COLLECTION_NAME)\n\n\n# ---------------------------------------------------------------------------\n# Request / response models\n# ---------------------------------------------------------------------------\n\nclass QueryRequest(BaseModel):\n    session_id: str\n    query: str\n    top_k: Optional[int] = 5\n\n\nclass QueryResponse(BaseModel):\n    answer: str\n    sources: List[Dict[str, Any]]\n\n\nclass UploadResponse(BaseModel):\n    session_id: str\n    files_processed: int\n    documents_extracted: int\n    chunks_indexed: int\n\n\n# ---------------------------------------------------------------------------\n# Routes\n# ---------------------------------------------------------------------------\n\n@app.get(\"/api/health\")\ndef health():\n    return {\"status\": \"ok\"}\n\n\n@app.post(\"/api/upload\", response_model=UploadResponse)\nasync def upload_files(\n    files: List[UploadFile] = File(...),\n    session_id: Optional[str] = Form(None),\n):\n    \"\"\"Accept one or more files, extract + describe their content in-memory,\n    chunk it, and index it into Weaviate under a session_id so concurrent\n    users never see each other's documents.\"\"\"\n    session_id = session_id or str(uuid.uuid4())\n\n    if not files:\n        raise HTTPException(status_code=400, detail=\"No files were uploaded.\")\n\n    file_payloads = []\n    for f in files:\n        content = await f.read()\n        file_payloads.append((f.filename, content))\n\n    try:\n        raw_documents = load_multimodal_data(file_payloads)\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=f\"Document processing failed: {e}\")\n\n    chunk_count = 0\n    client = get_weaviate_client()\n    try:\n        collection = ensure_collection(client)\n        with collection.batch.dynamic() as batch:\n            for doc in raw_documents:\n                metadata = doc.get(\"metadata\", {})\n                for chunk in chunk_text(doc.get(\"text\", \"\")):\n                    batch.add_object(properties={\n                        \"text\": chunk,\n                        \"source\": metadata.get(\"source\", \"unknown\"),\n                        \"doc_type\": metadata.get(\"type\", \"text\"),\n                        \"page_num\": int(metadata.get(\"page_num\") or 0),\n                        \"session_id\": session_id,\n                    })\n                    chunk_count += 1\n    finally:\n        client.close()\n\n    return UploadResponse(\n        session_id=session_id,\n        files_processed=len(files),\n        documents_extracted=len(raw_documents),\n        chunks_indexed=chunk_count,\n    )\n\n\n@app.post(\"/api/query\", response_model=QueryResponse)\ndef query_documents(payload: QueryRequest):\n    \"\"\"Retrieve the most relevant chunks for this session and ask the LLM\n    to answer the question grounded in them.\"\"\"\n    client = get_weaviate_client()\n    try:\n        if not client.collections.exists(COLLECTION_NAME):\n            raise HTTPException(\n                status_code=404,\n                detail=\"No documents have been indexed yet. Upload files first.\",\n            )\n        collection = client.collections.get(COLLECTION_NAME)\n        results = collection.query.near_text(\n            query=payload.query,\n            filters=Filter.by_property(\"session_id\").equal(payload.session_id),\n            limit=payload.top_k or 5,\n        )\n    finally:\n        client.close()\n\n    if not results.objects:\n        return QueryResponse(\n            answer=(\n                \"I couldn't find anything relevant in the documents uploaded \"\n                \"for this session. Try uploading files first, or rephrase \"\n                \"your question.\"\n            ),\n            sources=[],\n        )\n\n    context_parts = []\n    sources = []\n    for obj in results.objects:\n        props = obj.properties\n        context_parts.append(f\"[Source: {props.get('source')}] {props.get('text')}\")\n        sources.append({\n            \"source\": props.get(\"source\"),\n            \"type\": props.get(\"doc_type\"),\n            \"page_num\": props.get(\"page_num\"),\n        })\n\n    context = \"\\n\\n\".join(context_parts)\n    system_prompt = (\n        \"You are a meticulous financial research assistant. Answer the \"\n        \"user's question using ONLY the provided context extracted from \"\n        \"their uploaded financial documents. If the context doesn't contain \"\n        \"enough information, say so plainly rather than guessing. Reference \"\n        \"source names inline where it's useful.\"\n    )\n    user_prompt = f\"Context:\\n{context}\\n\\nQuestion: {payload.query}\"\n\n    try:\n        answer = chat_completion(system_prompt, user_prompt)\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=f\"LLM generation failed: {e}\")\n\n    return QueryResponse(answer=answer, sources=sources)\n\n\n@app.delete(\"/api/session/{session_id}\")\ndef clear_session(session_id: str):\n    \"\"\"Delete all indexed chunks belonging to a session (e.g. on 'Clear Chat').\"\"\"\n    client = get_weaviate_client()\n    try:\n        if client.collections.exists(COLLECTION_NAME):\n            collection = client.collections.get(COLLECTION_NAME)\n            collection.data.delete_many(\n                where=Filter.by_property(\"session_id\").equal(session_id)\n            )\n    finally:\n        client.close()\n    return {\"status\": \"cleared\", \"session_id\": session_id}\n",
    "requirements.txt": "fastapi==0.115.0\npython-multipart==0.0.9\npydantic==2.9.2\nweaviate-client==4.9.6\ngroq==0.11.0\npymupdf==1.24.10\npython-pptx==1.0.2\nPillow==10.4.0\ntabulate==0.9.0\npandas==2.2.3\n",
}

for name, content in files.items():
    path = os.path.join(APP_DIR, name)
    with open(path, "w") as f:
        f.write(content)
    print(f"wrote {path} ({len(content)} bytes)")

In [ ]:
# --- Cell 3: sanity-check the files we just wrote ---------------------------
import py_compile, os

APP_DIR = "/content/ledger_api"
for name in ["utils.py", "document_processors.py", "main.py"]:
    path = os.path.join(APP_DIR, name)
    py_compile.compile(path, doraise=True)
    print(f"{name}: syntax OK")
print("All source files compiled cleanly.")

In [ ]:
# --- Cell 4: load secrets (Colab Secrets pane, with a manual fallback) -----
import os
from getpass import getpass

try:
    from google.colab import userdata
    def get_secret(name):
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        return val
except ImportError:
    def get_secret(name):
        return None

def require_secret(name, is_password=True):
    val = get_secret(name) or os.environ.get(name)
    if not val:
        prompt = f"{name} not found in Colab secrets — paste it here: "
        val = getpass(prompt) if is_password else input(prompt)
    os.environ[name] = val
    return val

require_secret("GROQ_API_KEY")
require_secret("WEAVIATE_URL", is_password=False)
require_secret("WEAVIATE_API_KEY")
SHARED_SECRET = require_secret("LEDGER_SHARED_SECRET")
NGROK_AUTH_TOKEN = require_secret("NGROK_AUTH_TOKEN")

# Only needed if you want Cell 8 (GitHub push) to run.
GITHUB_PAT = get_secret("GITHUB_PAT") or os.environ.get("GITHUB_PAT")

print("Secrets loaded.")

In [ ]:
# --- Cell 5: import the app + apply the shared-secret middleware -----------
import sys
sys.path.insert(0, "/content/ledger_api")
os.chdir("/content/ledger_api")

import main as ledger_main
from fastapi import Request
from fastapi.responses import JSONResponse

app = ledger_main.app

@app.middleware("http")
async def require_shared_secret(request: Request, call_next):
    if request.url.path.startswith("/api/") and request.url.path != "/api/health":
        if request.headers.get("x-api-key") != SHARED_SECRET:
            return JSONResponse(status_code=401, content={"detail": "Missing or invalid X-Api-Key."})
    return await call_next(request)

print("App loaded, middleware attached.")

In [ ]:
# --- Cell 6: run uvicorn in the background + self-test locally -------------
import threading, time, urllib.request, json
import uvicorn

PORT = 8000

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Self-test: hit /api/health locally before opening the tunnel, so failures
# show up here instead of silently propagating to ngrok/GitHub.
try:
    resp = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=5)
    body = json.loads(resp.read())
    assert resp.status == 200 and body.get("status") == "ok"
    print("Local health check: PASS ->", body)
except Exception as e:
    raise RuntimeError(f"Local server failed its own health check: {e}")

In [ ]:
# --- Cell 7: open the ngrok tunnel + test it publicly -----------------------
from pyngrok import ngrok, conf
import urllib.request, json

conf.get_default().auth_token = NGROK_AUTH_TOKEN

tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url.replace("http://", "https://")
print("Public backend URL:", public_url)

# Self-test through the actual tunnel, matching what your frontend will hit.
try:
    resp = urllib.request.urlopen(f"{public_url}/api/health", timeout=10)
    body = json.loads(resp.read())
    assert resp.status == 200 and body.get("status") == "ok"
    print("Public health check via ngrok: PASS ->", body)
except Exception as e:
    raise RuntimeError(
        f"Tunnel is up but the public health check failed: {e}. "
        "Check the ngrok dashboard / firewall settings."
    )

In [ ]:
# --- Cell 8 (optional): push api-config.json to GitHub ----------------------
# Set PUSH_TO_GITHUB = True once you're happy with the tunnel above and want
# your frontend (index.html / vercel.json) to pick up the new URL.
PUSH_TO_GITHUB = False

REPO_OWNER = "sarahabumandil"
REPO_NAME = "Ledger"
CONFIG_PATH = "api-config.json"

if PUSH_TO_GITHUB:
    import base64, json, time, requests

    if not GITHUB_PAT:
        raise RuntimeError("GITHUB_PAT secret not set — add it in the Colab Secrets pane first.")

    API_URL = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{CONFIG_PATH}"
    HEADERS = {"Authorization": f"Bearer {GITHUB_PAT}", "Accept": "application/vnd.github+json"}

    def push_config(api_base: str):
        payload = {
            "api_base": api_base,
            "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        content_b64 = base64.b64encode(json.dumps(payload, indent=2).encode()).decode()

        existing = requests.get(API_URL, headers=HEADERS)
        sha = existing.json().get("sha") if existing.status_code == 200 else None

        body = {"message": f"chore: update live API endpoint ({api_base})", "content": content_b64, "branch": "main"}
        if sha:
            body["sha"] = sha

        resp = requests.put(API_URL, headers=HEADERS, json=body)
        if resp.status_code not in (200, 201):
            raise RuntimeError(f"GitHub update failed: {resp.status_code} {resp.text}")
        print("api-config.json updated:", resp.json()["content"]["html_url"])

    push_config(public_url)
else:
    print("Skipped (PUSH_TO_GITHUB is False). Set it to True and re-run this cell when ready.")

In [ ]:
# --- Cell 9: keep the runtime alive + re-push if ngrok rotates --------------
import time
from pyngrok import ngrok

last_url = public_url
try:
    while True:
        time.sleep(60)
        current_tunnels = ngrok.get_tunnels()
        if current_tunnels:
            current_url = current_tunnels[0].public_url.replace("http://", "https://")
            if current_url != last_url:
                print("ngrok URL changed:", current_url)
                if PUSH_TO_GITHUB:
                    push_config(current_url)
                last_url = current_url
except KeyboardInterrupt:
    print("Stopped. Tunnel + server will die with this runtime.")